# Zipformer CS (VI+EN) — Fine-tuning & Evaluation

**GPU runtime.** Restores the artifacts produced by `data_process_colab.ipynb` from the Hugging Face Hub, then runs **fine-tune → before/after WER**.

Shared config lives in `colab/colab_config.yaml`. Run `data_process_colab.ipynb` first (on a CPU runtime) to build + push the data.

## 0. Setup

In [ ]:
# Python deps for training (GPU). k2 is the fragile one: it must match Colab's
# torch + CUDA. PyYAML is preinstalled on Colab but pinned for other runtimes.
import torch
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)

!pip -q install lhotse datasets soundfile sentencepiece wordfreq kaldialign \
  num2words huggingface_hub mlflow pyyaml
# k2: pick the wheel matching the torch/cuda above. The -f index lists builds.
!pip -q install k2 -f https://k2-fsa.github.io/k2/cuda.html || \
  echo 'k2 install failed - open https://k2-fsa.github.io/k2/cuda.html and pick the wheel'
!pip -q install -e icefall

import k2, lhotse, datasets
print('k2:', k2.__version__, '| lhotse:', lhotse.__version__, '| datasets:', datasets.__version__)

In [ ]:
# --- Load shared config -----------------------------------------------------
# Single edit point for BOTH notebooks: colab/colab_config.yaml.
import os, yaml

CONFIG_PATH = 'colab/colab_config.yaml'   # relative to the repo root
with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

REPO       = cfg['repo']
DRIVE      = cfg['drive']
PREFIX     = cfg['prefix']
BPE_MODEL  = cfg['bpe_model']
PRETRAINED = cfg['pretrained']
TEXT_NORM  = cfg['text_norm']
HF_DATA_REPO  = cfg['hf_data_repo']
HF_MODEL_REPO = cfg['hf_model_repo']
FORCE_RECOMPUTE = bool(cfg['force_recompute'])

# Derived paths (kept identical across both notebooks).
ASR      = f'{REPO}/icefall/egs/librispeech/ASR'
FBANK    = 'data/fbank'
EXP_DIR  = 'data/exp_finetune'
COMBINED = 'data/combined_dataset'
MANIFEST = 'data/combined_manifest.jsonl'
HF_SYNC  = os.path.abspath(f'{REPO}/colab/hf_sync.py')  # absolute: survives %cd

os.makedirs(DRIVE, exist_ok=True)
assert os.path.isdir(REPO), f'Put this repo at {REPO} (clone or upload).'
print('REPO =', REPO, '| DRIVE =', DRIVE, '| FORCE_RECOMPUTE =', FORCE_RECOMPUTE)

In [ ]:
# Place the repo's maintained scripts where the recipe expects them.
import shutil
shutil.copy(f'{REPO}/finetune.py', f'{ASR}/zipformer/finetune.py')
shutil.copy(f'{REPO}/decode.py', f'{ASR}/zipformer/decode.py')
shutil.copy(f'{REPO}/ctc_decode.py', f'{ASR}/zipformer/ctc_decode.py')
print('finetune/decode scripts placed under', ASR)

## 0b. Hugging Face Hub — restore artifacts

Log in, then pull the fbank cuts/feats (incl. MUSAN), the combined dataset, and the training inputs (bpe.model + base ckpt). Re-run after a restart to rehydrate the local disk.

In [ ]:
# Auth once per session (or set os.environ['HF_TOKEN'] = '...').
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
%cd {ASR}
import os
# Restore data artifacts (fbank cuts/feats + MUSAN + combined dataset).
if HF_DATA_REPO and not os.path.exists(f'{FBANK}/{PREFIX}_cuts_train.jsonl.gz'):
    !python {HF_SYNC} pull --repo-id {HF_DATA_REPO} --repo-type dataset --local data --path-in-repo combined_dataset
    !python {HF_SYNC} pull --repo-id {HF_DATA_REPO} --repo-type dataset --local data --path-in-repo fbank
# Restore training inputs (bpe.model + base ckpt).
if HF_MODEL_REPO and not os.path.exists(PRETRAINED):
    !python {HF_SYNC} pull --repo-id {HF_MODEL_REPO} --repo-type model --local model \
      --allow-pattern 'bpe.model' --allow-pattern 'epoch-35-avg-*.pt'
print('train cuts present:', os.path.exists(f'{FBANK}/{PREFIX}_cuts_train.jsonl.gz'))
print('base ckpt present:', os.path.exists(PRETRAINED))

## 6. Fine-tune (GPU, resumable)

Checkpoints write to `EXP_DIR` locally. The Resume cell pulls the latest `epoch-N.pt` from the Hub and sets `START_EPOCH`/`DO_FINETUNE` automatically.

In [ ]:
# MLflow on Databricks (optional): metrics + best weight pushed to your workspace.
# Leave USE_MLFLOW = 0 to skip. Token is read via getpass - never hardcode it.
import os, getpass

USE_MLFLOW = 1  # set 0 to disable MLflow logging

if USE_MLFLOW:
    os.environ['MLFLOW_TRACKING_URI'] = 'databricks'
    os.environ['DATABRICKS_HOST'] = 'https://<your-workspace>.cloud.databricks.com'  # <- EDIT
    if not os.environ.get('DATABRICKS_TOKEN'):
        os.environ['DATABRICKS_TOKEN'] = getpass.getpass('Databricks token: ')
    MLFLOW_EXPERIMENT = '/Users/taiinguyenn139@gmail.com/zipformer-cs-finetune'  # <- EDIT
    MLFLOW_RUN_NAME   = PREFIX
    MLFLOW_ARGS = (f'--use-mlflow 1 --mlflow-experiment {MLFLOW_EXPERIMENT} '
                   f'--mlflow-run-name {MLFLOW_RUN_NAME}')
else:
    MLFLOW_ARGS = '--use-mlflow 0'
print('MLFLOW_ARGS =', MLFLOW_ARGS)

In [ ]:
# MUSAN noise augmentation config. ENABLE_MUSAN=0 to skip.
# MUSAN_SOURCE: 'full' (music+noise+speech) or 'speech' (human background only).
ENABLE_MUSAN = 1
MUSAN_SOURCE = 'full'      # 'full' | 'speech'
SNR_LOW, SNR_HIGH = 5, 20  # lower = harsher background

_musan_cuts = {'full': 'musan_cuts.jsonl.gz',
               'speech': 'musan_speech_cuts.jsonl.gz'}[MUSAN_SOURCE]
if ENABLE_MUSAN:
    MUSAN_ARGS = (f'--enable-musan 1 --musan-cuts {_musan_cuts} '
                  f'--musan-snr-low {SNR_LOW} --musan-snr-high {SNR_HIGH}')
else:
    MUSAN_ARGS = '--enable-musan 0'
print('MUSAN_ARGS =', MUSAN_ARGS)

In [ ]:
# CTC head config. Set USE_CTC=1 if you fine-tune / evaluate the CTC head
# (joint CTC+transducer). Threads --use-ctc through fine-tune and the WER decode.
USE_CTC = 1
CTC_ARGS = f'--use-ctc {USE_CTC}'
print('CTC_ARGS =', CTC_ARGS)

In [ ]:
# Smoke test: 1 epoch, tiny batch. Confirms ckpt loads + a checkpoint is written.
%cd {ASR}
!python zipformer/finetune.py \
  --world-size 1 --num-epochs 1 --start-epoch 1 --use-fp16 1 \
  --do-finetune 1 --finetune-ckpt {PRETRAINED} --base-lr 0.0045 --use-mux 0 \
  --bpe-model {BPE_MODEL} --exp-dir {EXP_DIR} \
  --manifest-dir {FBANK} \
  --train-manifest {PREFIX}_cuts_train.jsonl.gz \
  --valid-manifest {PREFIX}_cuts_dev.jsonl.gz \
  --valid-set-name {PREFIX} \
  {MLFLOW_ARGS} \
  {CTC_ARGS} \
  {MUSAN_ARGS} --max-duration 80

In [ ]:
%cd {ASR}
# Resume: pull the latest epoch-N.pt from the Hub, derive --start-epoch.
START_EPOCH, DO_FINETUNE = 1, 1
if HF_MODEL_REPO:
    out = !python {HF_SYNC} pull-latest-ckpt --repo-id {HF_MODEL_REPO} --local {EXP_DIR}
    last = next((l.strip() for l in reversed(out) if l.strip().isdigit()), '')
    if last:
        START_EPOCH, DO_FINETUNE = int(last) + 1, 0
        print(f'Resuming from epoch {last} -> --start-epoch {START_EPOCH}')
    else:
        print('No checkpoint on the Hub - starting fresh.')
print('START_EPOCH =', START_EPOCH, '| DO_FINETUNE =', DO_FINETUNE)

In [ ]:
# Live training monitor. finetune.py logs to {EXP_DIR}/tensorboard by default.
# Launch BEFORE the full run below; it refreshes as training writes scalars.
%cd {ASR}
%load_ext tensorboard
%tensorboard --logdir {EXP_DIR}/tensorboard

In [ ]:
# Full run. START_EPOCH / DO_FINETUNE come from the Resume cell above
# (fresh: 1 / 1; resuming: latest_epoch+1 / 0).
%cd {ASR}
!python zipformer/finetune.py \
  --world-size 1 --num-epochs 20 --start-epoch {START_EPOCH} --use-fp16 1 \
  --do-finetune {DO_FINETUNE} --finetune-ckpt {PRETRAINED} --base-lr 0.0045 --use-mux 0 \
  --bpe-model {BPE_MODEL} --exp-dir {EXP_DIR} \
  --manifest-dir {FBANK} \
  --train-manifest {PREFIX}_cuts_train.jsonl.gz \
  --valid-manifest {PREFIX}_cuts_dev.jsonl.gz \
  --valid-set-name {PREFIX} \
  {MLFLOW_ARGS} \
  {CTC_ARGS} \
  {MUSAN_ARGS} --max-duration 300

In [ ]:
# On-demand: push checkpoints written so far to the model repo.
# Re-run anytime (e.g. before the session times out) to back up progress.
if HF_MODEL_REPO:
    !python {HF_SYNC} push --repo-id {HF_MODEL_REPO} --repo-type model --private \
      --local {EXP_DIR} --allow-pattern 'epoch-*.pt'
else:
    print('hf_model_repo empty - skipping checkpoint push.')

## 7. Before/after WER (GPU)

Decode the **base** checkpoint and the **fine-tuned** weight(s) on the held-out `test` split with greedy search, then compare WER. Same cuts, same BPE, so the numbers are directly comparable. A base WER near 100% means a BPE/`text_norm` mismatch, not a real baseline — fix that first.

In [ ]:
# Before/after WER on the held-out test split(s).
# Transducer head -> decode.py greedy_search; CTC head -> ctc_decode.py
# ctc-greedy-search (only when USE_CTC). SWEEP_ALL=False -> latest epoch.
%cd {ASR}
import glob, re, os

SWEEP_ALL = False

def run_transducer(checkpoint, out_dir, test_manifest):
    os.makedirs(out_dir, exist_ok=True)
    !python zipformer/decode.py \
      --checkpoint {checkpoint} --decoding-method greedy_search --use-ctc {USE_CTC} \
      --bpe-model {BPE_MODEL} --exp-dir {out_dir} \
      --manifest-dir {FBANK} --test-manifest {test_manifest} \
      --test-set-name {PREFIX} --max-duration 300

def run_ctc(checkpoint, out_dir, test_manifest):
    os.makedirs(out_dir, exist_ok=True)
    !python zipformer/ctc_decode.py \
      --checkpoint {checkpoint} --decoding-method ctc-greedy-search --use-ctc 1 \
      --bpe-model {BPE_MODEL} --exp-dir {out_dir} \
      --manifest-dir {FBANK} --test-manifest {test_manifest} \
      --test-set-name {PREFIX} --max-duration 300

def read_wer(out_dir, subdir):
    wers = []
    for fp in glob.glob(f'{out_dir}/{subdir}/wer-summary-{PREFIX}-*.txt'):
        for line in open(fp, encoding='utf8').read().splitlines()[1:]:  # skip header
            parts = line.split('\t')
            if len(parts) == 2:
                try:
                    wers.append(float(parts[1]))
                except ValueError:
                    pass
    return min(wers) if wers else None

# Test conditions: clean always; noisy if data notebook generated it.
conditions = [('clean', f'{PREFIX}_cuts_test.jsonl.gz')]
if os.path.exists(f'{FBANK}/{PREFIX}_cuts_test_noisy.jsonl.gz'):
    conditions.append(('noisy', f'{PREFIX}_cuts_test_noisy.jsonl.gz'))

# Checkpoints: base + latest (or all) fine-tuned epochs.
epochs = sorted(int(re.search(r'epoch-(\d+)\.pt', p).group(1))
                for p in glob.glob(f'{EXP_DIR}/epoch-*.pt'))
ckpts = [('base', PRETRAINED)]
for ep in (epochs if SWEEP_ALL else epochs[-1:]):
    ckpts.append((f'epoch-{ep}', f'{EXP_DIR}/epoch-{ep}.pt'))

# Heads: transducer always; ctc when USE_CTC. (runner, result subdir)
heads = [('transducer', run_transducer, 'greedy_search')]
if int(USE_CTC):
    heads.append(('ctc', run_ctc, 'ctc-greedy-search'))

results = {}
for cond, manifest in conditions:
    for label, ckpt in ckpts:
        for hname, runner, subdir in heads:
            out_dir = f'data/exp_wer_{label}_{cond}_{hname}'
            runner(ckpt, out_dir, manifest)
            results[(label, cond, hname)] = read_wer(out_dir, subdir)

cols = [(c, h) for c, _ in conditions for h, _, _ in heads]
print('\n=== WER% (greedy) on', PREFIX, 'test ===')
print(f'{"checkpoint":<14}' + ''.join(f'{c+"/"+h:>16}' for c, h in cols))
for label, _ in ckpts:
    row = ''.join(
        f'{("n/a" if results[(label,c,h)] is None else f"{results[(label,c,h)]:.2f}"):>16}'
        for c, h in cols)
    print(f'{label:<14}{row}')